# dogcat

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

repo_dir = Path("/kaggle/working/eX-LAViTs")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/VuTrinhNguyenHoang/eX-LAViTs.git", str(repo_dir)],
        check=True,
    )
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
print("Code:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())


In [ ]:
%pip install -q timm==1.0.30 scikit-learn psutil

In [ ]:
import numpy as np
import pandas as pd
import json
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from IPython.display import display
from src.models.linear_vit import create_linear_attention_vit
from src.utils.metrics import compute_loss_and_accuracy
from src.utils.revision_evaluation import (
    evaluate_methods,
    format_summary,
    save_metadata,
    benchmark_methods,
)


## Configuration

In [ ]:
DATASET = "dogcat"
model_dir = Path("path/to/dir")
data_dir = Path("/kaggle/input/cat-dog-images-for-classification")
output_dir = Path("/kaggle/working/results") / DATASET
output_dir.mkdir(parents=True, exist_ok=True)
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 0
MAX_SAMPLES = None
STEPS = 20
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(SEED)
torch.manual_seed(SEED)
print(
    "Device:", DEVICE, "GPU:", torch.cuda.get_device_name() if DEVICE.type == "cuda" else None
)
print("Output:", output_dir)


## Data

In [ ]:
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split

transform = T.Compose(
    [
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
    ]
)


class ImageDataset(Dataset):
    def __init__(self, frame):
        self.df = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        with Image.open(row["image"]) as image:
            x = transform(image.convert("RGB"))
        return x, int(row["labels"])


In [ ]:
df = pd.read_csv(data_dir / "cat_dog.csv")
df["image"] = df["image"].map(lambda value: str(data_dir / "cat_dog" / value))
class_names = [str(value) for value in sorted(df["labels"].unique())]

train_df, tmp_df = train_test_split(
    df, test_size=0.3, random_state=SEED, stratify=df["labels"]
)
valid_df, test_df = train_test_split(
    tmp_df, test_size=0.5, random_state=SEED, stratify=tmp_df["labels"]
)
test_dataset = ImageDataset(test_df)
sample_ids = test_dataset.df["image"].tolist()
test_dataset.df.to_csv(output_dir / "test_manifest.csv", index=False)
n_classes = len(class_names)
print("Train / validation / test:", len(train_df), len(valid_df), len(test_dataset))


## Model

In [ ]:
checkpoint_path = model_dir / f"{DATASET}.pth"
model = create_linear_attention_vit(
    "vit_tiny_patch16_224",
    pretrained=False,
    num_classes=n_classes,
    img_size=IMAGE_SIZE,
    verbose=False,
)
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
state = checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint
model.load_state_dict(state, strict=True)
del checkpoint, state
model.to(DEVICE).eval()
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))


In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda",
)
classification = compute_loss_and_accuracy(
    model, test_loader, nn.CrossEntropyLoss(), DEVICE, show_progress=True
)
print(classification)
(output_dir / "classification.json").write_text(json.dumps(classification, indent=2))


## Evaluation

In [ ]:
config = dict(
    dataset=DATASET,
    seed=SEED,
    image_size=IMAGE_SIZE,
    max_samples=MAX_SAMPLES,
    steps=STEPS,
    split_protocol="original stratified 70/15/15; seed 42; CSV order",
    class_names=class_names,
    data_dir=str(data_dir),
    model_name="vit_tiny_patch16_224",
    kernel="elu",
    eps=1e-6,
    normalization_mean=[0.485, 0.456, 0.406],
    normalization_std=[0.229, 0.224, 0.225],
)
metadata = save_metadata(output_dir, config, model, checkpoint_path)


In [ ]:
summary = evaluate_methods(
    model,
    test_dataset,
    DEVICE,
    output_dir,
    max_samples=MAX_SAMPLES,
    steps=STEPS,
    sample_ids=sample_ids,
)
display(format_summary(summary))


## Compute

In [ ]:
xb = test_dataset[0][0].unsqueeze(0).to(DEVICE)
compute = benchmark_methods(model, xb, output_dir)
display(compute)
print("Saved:", output_dir)
